In [17]:
!pip install pdfplumber nltk sastrawi -q

In [18]:
import os
import re
import zipfile
import pdfplumber
import pandas as pd
from google.colab import drive

In [19]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os

PROJECT_FOLDER = "/content/drive/MyDrive/CBR-PHI"

RAW_FOLDER = os.path.join(PROJECT_FOLDER, "data", "raw")
PROCESSED_FOLDER = os.path.join(PROJECT_FOLDER, "data", "processed")
RESULT_FOLDER = os.path.join(PROJECT_FOLDER, "data", "results")
EVAL_FOLDER = os.path.join(PROJECT_FOLDER, "data", "eval")

ZIP_FILE = os.path.join(RAW_FOLDER, "Sub 2 CPMK 3.zip")

print("ZIP :", ZIP_FILE)
print("ZIP ditemukan :", os.path.exists(ZIP_FILE))

ZIP : /content/drive/MyDrive/CBR-PHI/data/raw/Sub 2 CPMK 3.zip
ZIP ditemukan : True


In [21]:
EXTRACT_FOLDER = "/content/putusan_phi"

os.makedirs(EXTRACT_FOLDER, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_FOLDER)

print("ZIP berhasil diekstrak")

ZIP berhasil diekstrak


In [30]:
pdf_files = []

for root, dirs, files in os.walk(EXTRACT_FOLDER):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print(f"Jumlah PDF : {len(pdf_files)}")

print("\nContoh file:")
for pdf in pdf_files[:5]:
    print(pdf)

Jumlah PDF : 30

Contoh file:
/content/putusan_phi/Sub 2 CPMK 3/putusan_8_k_pdt.sus-phi_2025_20260624162912.pdf
/content/putusan_phi/Sub 2 CPMK 3/putusan_17_k_pdt.sus-phi_2025_20260623232750.pdf
/content/putusan_phi/Sub 2 CPMK 3/putusan_18_k_pdt.sus-phi_2025_20260624162916.pdf
/content/putusan_phi/Sub 2 CPMK 3/putusan_3_k_pdt.sus-phi_2025_20260624162941.pdf
/content/putusan_phi/Sub 2 CPMK 3/putusan_48_k_pdt.sus-phi_2025_20260624162953.pdf


In [24]:
TEXT_FOLDER = os.path.join(PROCESSED_FOLDER, "cases_text")

os.makedirs(TEXT_FOLDER, exist_ok=True)

print(TEXT_FOLDER)

/content/drive/MyDrive/CBR-PHI/data/processed/cases_text


In [31]:
import pdfplumber

with pdfplumber.open(pdf_files[0]) as pdf:
    print("Jumlah halaman:", len(pdf.pages))

    first_page = pdf.pages[0].extract_text()

    print("\n=== 500 Karakter Pertama ===\n")
    print(first_page[:500])

Jumlah halaman: 7

=== 500 Karakter Pertama ===

a
i
s
e
n
o
d
n
I
k
i
l
b
u
p
e
a
R
i
s
g e
n
n
o
u
d
g
n
A
I
h k
a i
l
m b
u
a
p
k
Direktori Putusan Mahkamah Agung Republik Indonesia
e
h a
putusan.mahkamahagung.go.id
R
a i
P U T U S A N s
M Nomor 8 K/Pdt.Sus-PHI/2025
g e
n
DEMI nKEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
u M A H K A M A H A G U N G o
memeriksa perkara perdata khusus perselisihan hubungan indudstrial pada
g
tingkat kasasi memutus sebagai berikut dalam perkara antara:
n
A
1. MUHAMMAD NASIR, bertempat tinggal di Jalan Paku
I



In [32]:
import pdfplumber

TEXT_FOLDER = os.path.join(PROCESSED_FOLDER, "cases_text")
os.makedirs(TEXT_FOLDER, exist_ok=True)

success = 0

for i, pdf_path in enumerate(pdf_files, start=1):

    try:
        text = ""

        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

        output_file = os.path.join(TEXT_FOLDER, f"case_{i:03d}.txt")

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(text)

        success += 1

        print(f"✓ case_{i:03d}.txt")

    except Exception as e:
        print(f"✗ Gagal: {os.path.basename(pdf_path)}")
        print(e)

print(f"\nBerhasil ekstrak {success} dari {len(pdf_files)} PDF")

✓ case_001.txt
✓ case_002.txt
✓ case_003.txt
✓ case_004.txt
✓ case_005.txt
✓ case_006.txt
✓ case_007.txt
✓ case_008.txt
✓ case_009.txt
✓ case_010.txt
✓ case_011.txt
✓ case_012.txt
✓ case_013.txt
✓ case_014.txt
✓ case_015.txt
✓ case_016.txt
✓ case_017.txt
✓ case_018.txt
✓ case_019.txt
✓ case_020.txt
✓ case_021.txt
✓ case_022.txt
✓ case_023.txt
✓ case_024.txt
✓ case_025.txt
✓ case_026.txt
✓ case_027.txt
✓ case_028.txt
✓ case_029.txt
✓ case_030.txt

Berhasil ekstrak 30 dari 30 PDF


In [33]:
txt_files = sorted(os.listdir(TEXT_FOLDER))

print("Jumlah TXT :", len(txt_files))
print(txt_files[:5])

Jumlah TXT : 30
['case_001.txt', 'case_002.txt', 'case_003.txt', 'case_004.txt', 'case_005.txt']


In [34]:
import re

def clean_text(text):

    # lowercase
    text = text.lower()

    # hapus watermark website
    text = re.sub(r"putusan\.mahkamahagung\.go\.id", " ", text)

    # hapus tulisan mahkamah agung
    text = re.sub(r"direktori putusan mahkamah agung republik indonesia", " ", text)

    # hapus karakter aneh
    text = re.sub(r"[^\w\s]", " ", text)

    # hapus angka halaman
    text = re.sub(r"\b\d+\b", " ", text)

    # rapikan spasi
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [35]:
for file in os.listdir(TEXT_FOLDER):

    path = os.path.join(TEXT_FOLDER, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    text = clean_text(text)

    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

print("Cleaning selesai")

Cleaning selesai


In [36]:
sample = os.path.join(TEXT_FOLDER, "case_001.txt")

with open(sample, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

a i s e n o d n i k i l b u p e a r i s g e n n o u d g n a i h k a i l m b u a p k e h a r a i p u t u s a n s m nomor k pdt sus phi g e n demi nkeadilan berdasarkan ketuhanan yang maha esa u m a h k a m a h a g u n g o memeriksa perkara perdata khusus perselisihan hubungan indudstrial pada g tingkat kasasi memutus sebagai berikut dalam perkara antara n a muhammad nasir bertempat tinggal di jalan paku i ujung lingkungan vii gang wakt u kelurahan tanah h k enam ratus kecamatan medan marelan kota medan a i provinsi sumatera utara l m satria putra pohban bertempat tinggal di jalan gaperta gang iuntim lingkungan ii nomor 191a a kelurahan helvetia timur kecamatan medan helvetia p k kota madya medan provinsi sumatera utara e h keduanya dalam hal ini memberi kuasa kepada ganda a r a pasaribu s h dan kawan kawan para advokat pada kantor i s m hukum misericordiasdomini beralamat di jalan kapten g e muslim gang simalungun nomor medan helvetia kota n nmadya medan provinsi sumatera utara berdasar

In [37]:
word_count = []

for file in os.listdir(TEXT_FOLDER):

    path = os.path.join(TEXT_FOLDER, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    word_count.append(len(text.split()))

df_validation = pd.DataFrame({
    "Document": sorted(os.listdir(TEXT_FOLDER)),
    "Words": word_count
})

df_validation.head()

,Document,Words
0,case_001.txt,3609
1,case_002.txt,3079
2,case_003.txt,4169
3,case_004.txt,4093
4,case_005.txt,5105


In [38]:
print("Jumlah Dokumen :", len(df_validation))
print("Rata-rata Kata :", int(df_validation["Words"].mean()))
print("Minimum Kata :", df_validation["Words"].min())
print("Maximum Kata :", df_validation["Words"].max())

Jumlah Dokumen : 30
Rata-rata Kata : 5842
Minimum Kata : 3079
Maximum Kata : 33396
